In [1]:
library(tidyverse)

getwd()
list.files()

── Attaching core tidyverse packages ────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ──────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "/Users/rdlt102/Predicting-NHL-26-27-Breakout-Players-"

[1] "anaconda_projects"          "cleandata.ipynb"           
[3] "data.ipynb"                 "Predict NHL Breakout 26-27"
[5] "README.md"                  "Untitled.ipynb"

In [2]:
files <- list.files (
    path = "Predict NHL Breakout 26-27",
    full.names = TRUE
    )

files

[1] "Predict NHL Breakout 26-27/nhl_outcomes.rds"
 [2] "Predict NHL Breakout 26-27/nhl_players.rds" 
 [3] "Predict NHL Breakout 26-27/NHL15-16.csv"    
 [4] "Predict NHL Breakout 26-27/NHL16-17.csv"    
 [5] "Predict NHL Breakout 26-27/NHL17-18.csv"    
 [6] "Predict NHL Breakout 26-27/NHL18-19.csv"    
 [7] "Predict NHL Breakout 26-27/NHL19-20.csv"    
 [8] "Predict NHL Breakout 26-27/NHL20-21.csv"    
 [9] "Predict NHL Breakout 26-27/NHL21-22.csv"    
[10] "Predict NHL Breakout 26-27/NHL22-23.csv"    
[11] "Predict NHL Breakout 26-27/NHL23-24.csv"    
[12] "Predict NHL Breakout 26-27/NHL24-25.csv"    
[13] "Predict NHL Breakout 26-27/NHL25-26.csv"    
[14] "Predict NHL Breakout 26-27/player_bios.csv"

In [3]:
nhl_raw <- map_dfr (
    files,
    ~ read_csv (.x, show_col_types = FALSE)
    )
dim(nhl_raw)

Warning message:
“One or more parsing issues, call `problems()` on your data frame
for details, e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message:
“One or more parsing issues, call `problems()` on your data frame
for details, e.g.:
  dat <- vroom(...)
  problems(dat)”


[1] 192880    162

In [4]:
invisible(problems(nhl_raw))

In [5]:
guess_encoding(
    "Predict NHL Breakout 26-27/player_bios.csv"
    )

encoding,confidence
<chr>,<dbl>
ASCII,1


In [6]:
null_check <- map_dfr(files, function(path) {
  
  raw_contents <- readBin(
    path,
    what = "raw",
    n = file.info(path)$size
  )
  
  tibble(
    source_file = basename(path),
    null_bytes = sum(raw_contents == as.raw(0))
  )
})

null_check |>
  filter(null_bytes > 0)

basename(files)

source_file,null_bytes
<chr>,<int>
nhl_outcomes.rds,552
nhl_players.rds,4939


[1] "nhl_outcomes.rds" "nhl_players.rds"  "NHL15-16.csv"     "NHL16-17.csv"    
 [5] "NHL17-18.csv"     "NHL18-19.csv"     "NHL19-20.csv"     "NHL20-21.csv"    
 [9] "NHL21-22.csv"     "NHL22-23.csv"     "NHL23-24.csv"     "NHL24-25.csv"    
[13] "NHL25-26.csv"     "player_bios.csv"

In [7]:
all_csv_files <- list.files (
    path = ".",
    pattern = "\\.csv$",
    full.names = TRUE,
    recursive = TRUE,
    ignore.case = TRUE
    )
basename(all_csv_files)

[1] "NHL15-16.csv"    "NHL16-17.csv"    "NHL17-18.csv"    "NHL18-19.csv"   
 [5] "NHL19-20.csv"    "NHL20-21.csv"    "NHL21-22.csv"    "NHL22-23.csv"   
 [9] "NHL23-24.csv"    "NHL24-25.csv"    "NHL25-26.csv"    "player_bios.csv"

In [8]:
skater_files <- all_csv_files[
  grepl(
    "^NHL",
    basename(all_csv_files),
    ignore.case = TRUE
  )
]

basename(skater_files)
length(skater_files)

[1] "NHL15-16.csv" "NHL16-17.csv" "NHL17-18.csv" "NHL18-19.csv" "NHL19-20.csv"
 [6] "NHL20-21.csv" "NHL21-22.csv" "NHL22-23.csv" "NHL23-24.csv" "NHL24-25.csv"
[11] "NHL25-26.csv"

[1] 11

In [9]:
nhl_raw <- map_dfr(
  skater_files,
  ~ read_csv(.x, show_col_types = FALSE)
)

dim(nhl_raw)

[1] 50575   154

In [10]:
nhl_players <- nhl_raw |>
filter (situation == "all")

In [11]:
dim(nhl_players)

[1] 10115   154

In [12]:
nhl_players |>
count (season) |>
arrange (season)

season,n
<dbl>,<int>
2015,898
2016,887
2017,890
2018,906
2019,883
2020,913
2021,1003
2022,951
2023,924


In [13]:
nhl_players |>
select (
    name,
    season,
    team,
    position,
    games_played,
    I_F_goals,
    I_F_primaryAssists,
    I_F_secondaryAssists,
    I_F_points
    ) |>
head(20)

name,season,team,position,games_played,I_F_goals,I_F_primaryAssists,I_F_secondaryAssists,I_F_points
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Mikko Koivu,2015,MIN,C,82,17,18,21,56
Trevor van Riemsdyk,2015,CHI,D,82,3,7,4,14
Kyle Quincey,2015,DET,D,47,4,0,7,11
Filip Forsberg,2015,NSH,L,82,33,14,17,64
Brian Campbell,2015,FLA,D,82,6,12,13,31
Chris Terry,2015,CAR,L,68,8,2,1,11
Kyle Baun,2015,CHI,R,2,0,0,0,0
James van Riemsdyk,2015,TOR,L,40,14,10,5,29
Scott Harrington,2015,TOR,D,15,0,0,1,1


In [14]:
saveRDS (
    nhl_players,
    "Predict NHL Breakout 26-27/nhl_players.rds"
    )